# Random forest

# Load libraries & data

In [1]:
import pandas as pd

In [2]:
rna = pd.read_csv("data/RNASeq.csv")

metadata = pd.read_csv("data/metadata.csv")


In [3]:
# Use sample names as rownames in metadata
metadata = metadata.set_index("patient_id", drop=False)

# Reformat sample names
# cnv["sample_id"] = cnv["sample_id"].str.replace(".", "-", regex=False)
rna["sample_id"] = rna["sample_id"].str.replace(".", "-", regex=False)
# meth["sample_id"] = meth["sample_id"].str.replace(".", "-", regex=False)

def shorten_tcga_id(x):
    
    parts = x.split("-")
    # TCGA barcode structure: sample-level ends at element 4
    # if there are more parts → strip last two
    if len(parts) > 4:
        return "-".join(parts[:4])
    return x

# cnv["sample_id"] = cnv["sample_id"].apply(shorten_tcga_id)
rna["sample_id"] = rna["sample_id"].apply(shorten_tcga_id)
# meth["sample_id"] = meth["sample_id"].apply(shorten_tcga_id)

In [4]:
df = rna
metadata_rna_bool = metadata["patient_id"].isin(rna["sample_id"])
metadata_rna = metadata[metadata_rna_bool]
print(df.shape)
print(metadata_rna.shape)

(358, 14435)
(358, 7)


# Use pre-selected genes

In [5]:
topgenes_multiomics = pd.read_csv("results/top_genes_multiomics.csv", sep = ";")
topgenes_multiomics

,GEX,METH,CNV
0,MIF,cg23097686,FAM174A
1,DGCR5,cg04456219,ST8SIA4
2,EEF1G,cg02326386,SACM1L
3,AQP2,cg01702055,SLCO4C1
4,UBD,cg11201447,SLC25A46
...,...,...,...
495,HMGCR,cg03584506,CCL5
496,ATP6V1A,cg06613738,RDM1
497,MTHFS,cg05471495,MMP28
498,HINT2,cg24390590,GAS2L2


In [6]:
gex_genes = topgenes_multiomics["GEX"].tolist()
cols = ["sample_id"] + list(df.columns.intersection(gex_genes))
df = df[cols]
df


,sample_id,POLDIP2,KDM1A,SLC4A1,RHBDD2,ADIPOR2,TFAP2B,PSMC4,MATR3,RALBP1,...,TAS2R20,TRIM34,RNASE4,CORO7,RPL17,NBPF10,PIP4K2B,CCL14,ACACA,ADORA3
0,TCGA-AK-3458,0.470588,0.141620,0.181182,0.337352,0.208896,0.126446,0.233562,0.915471,0.154784,...,0.924894,0.903347,0.863992,0.651355,0.993972,0.425830,0.210836,0.806693,0.049124,0.755214
1,TCGA-B0-5711,0.240144,0.175085,0.101642,0.211876,0.309083,0.000693,0.166147,0.940761,0.342063,...,0.371233,0.928220,0.810919,0.775722,0.889143,0.889212,0.349269,0.769694,0.157556,0.723134
2,TCGA-B0-5696,0.507725,0.159357,0.212291,0.393265,0.258643,0.122497,0.331532,0.935010,0.240213,...,0.690570,0.840712,0.872722,0.833229,0.867872,0.633617,0.250676,0.851867,0.112520,0.741149
3,TCGA-CJ-4882,0.367976,0.051479,0.150211,0.504261,0.138156,0.041571,0.166355,0.739278,0.115707,...,0.894963,0.874316,0.816947,0.951916,0.927666,0.687452,0.262454,0.872514,0.075660,0.782928
4,TCGA-B0-5109,0.594679,0.180420,0.112312,0.345112,0.327028,0.191021,0.576249,0.895309,0.332571,...,0.425760,0.840019,0.477655,0.834546,0.916095,0.585810,0.236333,0.627105,0.225317,0.603062
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
353,TCGA-CJ-5680-11A,0.917481,0.821382,0.852006,0.887480,0.838426,0.372064,0.842999,0.221506,0.911730,...,0.113282,0.050509,0.207580,0.352318,0.198365,0.227257,0.879789,0.163029,0.760202,0.236749
354,TCGA-CW-5587-11A,0.959537,0.818402,0.983649,0.888104,0.849442,0.694034,0.847572,0.211529,0.948105,...,0.092496,0.085914,0.281230,0.281023,0.226564,0.225456,0.861290,0.179796,0.739001,0.174669
355,TCGA-CZ-5468-11A,0.939860,0.861290,0.962932,0.934248,0.891499,0.697984,0.843899,0.243608,0.973533,...,0.126308,0.057299,0.143629,0.316636,0.258089,0.208203,0.856024,0.108016,0.784037,0.196078
356,TCGA-B0-5711-11A,0.950807,0.831567,0.960299,0.884986,0.871752,0.812929,0.812028,0.216448,0.922677,...,0.073720,0.045936,0.151458,0.268136,0.214162,0.200859,0.872722,0.130881,0.782304,0.170651


# Split test & training data

In [7]:
# Split training & test data
from sklearn.model_selection import train_test_split

X = df.drop(columns=["sample_id"])
y = metadata_rna["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,        # key line
    random_state=42
)

# Random Forest for RNAseq data

In [8]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_selection import VarianceThreshold

pipeline = Pipeline([
    ("rf", RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1))
])

In [9]:
# Train the model
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('rf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If flo

In [10]:

# extract fitted RF model
rf = pipeline.named_steps["rf"]

# feature importance
importances = rf.feature_importances_

# create sorted dataframe
feat_imp = pd.DataFrame({
    "feature": X_train.columns,
    "importance": importances
}).sort_values("importance", ascending=False)

print(feat_imp)

      feature  importance
427     RBM34       0.012
95   ARHGEF10       0.010
156     PRDX6       0.010
322       VCP       0.010
153      SMG7       0.010
..        ...         ...
259     SURF4       0.000
268     FXYD4       0.000
270     RGS18       0.000
274     KCNJ1       0.000
499    ADORA3       0.000

[500 rows x 2 columns]


In [13]:
feat_imp.to_csv("results/RNAseq_imp.csv", index = False)

In [11]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        14
           1       1.00      1.00      1.00        58

    accuracy                           1.00        72
   macro avg       1.00      1.00      1.00        72
weighted avg       1.00      1.00      1.00        72

ROC-AUC: 1.0
